# Exercise 1 — OpsTask and TaskStore

`OpsTask` is a simple dataclass representing one unit of operational work.  `TaskStore` is an in-memory registry that assigns auto-ids, tracks status transitions, and supports filtering by status.

In [ ]:
import json
from dataclasses import dataclass, field

def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    text = str(text)
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except Exception:
        return None

# ── Exercise: implement OpsTask and TaskStore ─────────────────────────────────

@dataclass
class OpsTask:
    """A single unit of ops work.  status: pending/running/done/failed."""
    id:          str
    title:       str
    description: str = ""
    status:      str = "pending"
    result:      str = ""


class TaskStore:
    """In-memory registry of OpsTask objects with auto-incrementing ids."""

    def __init__(self):
        self._tasks   = {}
        self._counter = 0

    def add(self, title, description=""):
        # TODO: increment counter, create id "task_001" etc., create OpsTask,
        # store in self._tasks, return the task
        return OpsTask(id="task_001", title=title, description=description)

    def get(self, task_id):
        # TODO: return self._tasks.get(task_id)
        return None

    def all(self):
        # TODO: return list(self._tasks.values())
        return []

    def pending(self):
        # TODO: return tasks where status == "pending"
        return []

    def update(self, task_id, status, result=""):
        # TODO: look up task, set status (and result if non-empty), return task or None
        return None

    def __len__(self):
        # TODO: return number of tasks
        return 0


### Checks

In [ ]:
checks = 0

# 1 — OpsTask constructs with defaults
try:
    t = OpsTask(id="t1", title="Check disk")
    assert t.status == "pending" and t.result == "" and t.description == ""
    checks += 1; print("✅ 1 OpsTask constructs with correct defaults")
except Exception as e:
    print("❌ 1:", e)

# 2 — TaskStore.add returns OpsTask with auto id
try:
    store = TaskStore()
    t1 = store.add("Lint check")
    t2 = store.add("Test run", "run pytest")
    assert t1.id == "task_001" and t2.id == "task_002"
    assert t1.title == "Lint check" and t2.description == "run pytest"
    checks += 1; print("✅ 2 TaskStore.add assigns auto ids correctly")
except Exception as e:
    print("❌ 2:", e)

# 3 — TaskStore.get retrieves by id
try:
    store = TaskStore()
    t = store.add("Check logs")
    assert store.get(t.id) is t
    assert store.get("task_999") is None
    checks += 1; print("✅ 3 TaskStore.get retrieves task by id")
except Exception as e:
    print("❌ 3:", e)

# 4 — TaskStore.pending returns only pending tasks
try:
    store = TaskStore()
    t1 = store.add("Task A")
    t2 = store.add("Task B")
    store.update(t1.id, "done")
    pending = store.pending()
    assert len(pending) == 1 and pending[0].id == t2.id
    checks += 1; print("✅ 4 TaskStore.pending returns only pending tasks")
except Exception as e:
    print("❌ 4:", e)

# 5 — TaskStore.update changes status and result
try:
    store = TaskStore()
    t = store.add("Deploy")
    store.update(t.id, "done", "Deployed v1.2")
    assert t.status == "done" and t.result == "Deployed v1.2"
    assert len(store) == 1
    checks += 1; print("✅ 5 TaskStore.update changes status and result; __len__ works")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
